In [ ]:
from llama_cloud import LlamaCloud, AsyncLlamaCloud
import os
from langgraph.graph import StateGraph, END, START
from langchain_openai import ChatOpenAI
from typing import TypedDict, List, Dict, Any
from IPython.display import Image, display
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, AnyMessage
from pathlib import Path
from typing_extensions import Annotated

In [ ]:
general_knowledge = """
PDF(pair distribution function) analysis is a powerful technique used in materials science to study the local atomic structure.

diffpy.cmi is a comprehensive library for complex modeling and analysis of diffraction data, including PDF analysis.

diffpy.srfit is the backend for fitting and refining structural models in diffpy.cmi.

The general idea for refinement using diffpy.srfit is:
1. Use diffpy.srfit.fitbase.FitRecipe to manage all parameters. It picks up all parameters from:
    1. diffpy.srfit.pdf.PDFGenerator(s) for structural parameters.
    2. diffpy.srfit.fitbase.FitContribution(s) for special effect parameters.
2. Use diffpy.srfit.fitbase.FitContribution to hold 
    1. the corresponding relationship between one experiment PDF dataset and the corresponding structure model(s).
    2. the special effect parameters to be considered during the refinement, e.g., scaling factor, nanoparticle effect, etc.
3. Use diffpy.srfit.pdf.PDFGenerator to build a parameterized PDF model from a standard structure file (e.g., .cif file).
4. Use diffpy.srfit.fitbase.Profile to hold the experiment PDF data.

One FitRecipe corresponds to
one or multiple FitContribution(s) depends on the number of experiment PDF datasets and

One PDFContribution corresponds to
one or multiple PDFGenerator(s) depends on the number of structure models involved in the experiment PDF and
One Profile

One PDFGenerator corresponds to
one structure model with many structural parameters.

One structure model corresponds to
one standard structure file (e.g., .cif file) with possible modifications.

One Profile corresponds to
one experiment PDF dataset.

Before Human experts write a refinement script using diffpy.srfit, they typically consider the following aspects:
1. the number of PDF datasets to be refined against.
2. the avaiailable structure files for each phase present in each experiment PDF data.
3. the effects to be considered during the refinemnt: scaling factor, nanoparticle effect, etc.
4. the allowed variations of the structure model parameters during the refinement determined by constraints and restrains.

When Human experts write a refinement script using diffpy.srfit, they typically follow these steps:
1. Load all experiment PDF data as the diffpy.srfit.fitbase.Profile object(s).
    1. When multiple datasets are present, multiple diffpy.srfit.fitbase.Profile objects are created.
2. Load corresponding standard structure file(s) and created the diffpy.srfit.pdf.PDFGenerator obeject(s) for each diffpy.srfit.fitbase.Profile object.
    1. When multiple phase are present in one PDF dataset, multiple PDFGenerator objects are created accordingly in that dataset.
    2. When the standard structure file doesn't fully represent the actual structure, modify the internal diffpy.structure.Structure accordingly before creating the PDFGenerator object.
3. Create a diffpy.srfit.fitbase.FitContribution object to link the PDFGenerator(s) and the corresponding Profile for each diffpy.srfit.fitbase.Profile object.
    1. Usually scaling factor are considered in this stage.
    2. When nanoparticle effect are present, it's effect is also handled in this stage.
    3. When multiple PDF datasets are present, multiple diffpy.srfit.fitbase.FitContribution objects are created accordingly.
4. Create a diffpy.srfit.fitbase.FitRecipe object to manage the overall refinement process.
    1. When there are special constraints (e.g., force two variables in different contributions to be equal), set up the constraints in this stage.
    2. When there are restrains (e.g., bond length restration, ratio is between 0 and 1), set up the restrains in this stage.
"""

In [ ]:
generate_queries_prompt = f"""
Background knowledge:
{general_knowledge}

Instructions:
1. You are an expert at generating concise and relevant queries to gather information for code generation tasks.
2. You should understand the user's intension, adapt user's request and generate relevant and concise 'How to' queries.
3. If the user's intension is unclear, or it is not on the provided scope , you should respond with "N/A".
4. The scope of the information is limited to 
    1. PDF (pair distribution function) data refinement using diffpy.srfit/diffpy.cmi.
    2. Four sections of the refinement script using diffpy.srfit as described below
    3. The trivial case and variations of each section as described below.
5. The script is usually composed of four sections in this order:
    - initialize the data profile
    - initialize the structure model, and create the PDFGenerator object
    - initialize the contribution
    - initialize the fit recipe

Description of supported features/variations for each section:
1. Initialize the data profile:
    - trivial single data profile
2. Initialize the structure model:
    - single-phase structure model (trivial)
    - multi-phase structure model
3. Initialize the contribution:
    - single contribution (trivial)
    - multiple contributions
4. Initialize the fit recipe:
    - trivial fit recipe
If it is not specified in the user's request, you should assume the trivial version of each section.

Return exactly this structure:

{{
  "profile_queries" : "content",
  "structure_queries" : "content",
  "contribution_queries" : "content",
  "recipe_queries" : "content"
}}
"""
generate_queries_system_prompt = SystemMessage(content=generate_queries_prompt)

In [48]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
msg = HumanMessage(content="I want to fit one PDF data that have two phase using diffpy.srfit.")
response = llm.invoke([generate_queries_system_prompt, msg])

'```json\n{\n    "initialize_data_profile_queries": "How to load a single PDF dataset as a diffpy.srfit.fitbase.Profile object?",\n    "initialize_structure_model_queries": "How to create multiple PDFGenerator objects for a two-phase structure model using standard structure files?",\n    "initialize_contribution_queries": "How to create multiple diffpy.srfit.fitbase.FitContribution objects to link the PDFGenerators to the Profile?",\n    "initialize_fit_recipe_queries": "How to create a trivial diffpy.srfit.fitbase.FitRecipe object to manage the refinement process?"\n}\n```'

In [ ]:
def string_reducer(left, right) -> str:
    return left + "\n" + right

class State(TypedDict):
    """The state schema defining information passed between nodes."""
    skeleton : Annotated[str, string_reducer]
    script: Annotated[str, string_reducer]
    context = {
        "parallarization": "",
        "profile": "",
        "structure": "",
        "contribution": "",
        "recipe": ""
    }
    messages : List[AnyMessage]

def generate_fetch_queries(state):
    """Generate the queries text given the user's request that fetch 
    documentation for parallelization, profile ,structure, contribution ,recipe
    initialization stages."""
    pass

def fetch_context(state):
    """Fetch the context for each of the stages based on the queries."""
    pass

def compose_script(state):
    """Compose the final script based on the skeleton, context, and messages."""
    pass
    
    